# Tema 3 - Amplificadores de Una Etapa BJT y Combinacion de Etapas

**Electronica General - 2o GIERM**

---

## Objetivos de aprendizaje

- Dominar el modelo de pequena senal del BJT y sus parametros ($r_\pi$, $g_m$, $r_o$)
- Analizar el amplificador **Emisor Comun (CE)** y calcular $A_v$, $r_{in}$, $r_{out}$
- Analizar el amplificador **Colector Comun (CC)** y comprender su papel como buffer
- Comparar sistematicamente las configuraciones BJT frente a las equivalentes MOS
- Disenar combinaciones de etapas: **Cascode** (CE+CB) y **Buffer** (CE+CC)
- Resolver ejercicios completos con valores tipicos del BJT

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch
from matplotlib.lines import Line2D

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.labelsize'] = 13
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['figure.dpi'] = 100

COLOR_PRINCIPAL = '#2171b5'
COLOR_FIJO = '#636363'
COLOR_PUNTO = '#238b45'
COLOR_ROJO = '#cb181d'
COLOR_NARANJA = '#ff7f00'
COLOR_MORADO = '#6a3d9a'
COLOR_CLARO = '#a6cee3'

print('Configuracion lista.')

---

## 1. Modelo de Pequena Senal del BJT

El BJT en zona activa directa se linealiza alrededor del **punto de operacion** $(I_C, V_{CE})$.
El modelo hibrido-$\pi$ tiene tres parametros fundamentales:

| Parametro | Expresion | Significado |
|-----------|-----------|-------------|
| $g_m$ | $\dfrac{I_C}{V_T}$ | Transconductancia |
| $r_\pi$ | $\dfrac{\beta}{g_m} = \dfrac{V_T}{I_B}$ | Resistencia de entrada (base-emisor) |
| $r_o$ | $\dfrac{V_A}{I_C}$ | Resistencia de salida (efecto Early) |

> **Diferencia clave con MOS:** El BJT tiene $r_\pi$ **finita** (tipicamente $\sim 1\,k\Omega$),
> mientras que el MOSFET tiene impedancia de entrada **infinita** ($r_{gs} \to \infty$).

In [ ]:
# === Parametros tipicos del BJT ===
IC = 100e-6      # Corriente de colector (A)
Vt = 26e-3        # Voltaje termico (V) a T=300K
beta = 100        # Ganancia de corriente
VA = 2.0          # Voltaje Early (V)
RS = 0.1e3        # Resistencia de fuente (ohm)
RL = 5e3          # Resistencia de carga (ohm)

# Calculos
gm1 = IC / Vt
rpi1 = beta / gm1
ro1 = VA / IC

print('=== Parametros de Pequena Senal del BJT ===')
print(f'IC  = {IC*1e6:.0f} uA')
print(f'Vt  = {Vt*1e3:.0f} mV')
print(f'beta = {beta}')
print(f'VA  = {VA:.1f} V')
print(f'---')
print(f'gm1  = {gm1*1e3:.3f} mA/V')
print(f'rpi1 = {rpi1*1e-3:.3f} kohm')
print(f'ro1  = {ro1*1e-3:.1f} kohm')
print(f'RS   = {RS*1e-3:.1f} kohm')
print(f'RL   = {RL*1e-3:.1f} kohm')

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(12, 7))
ax.set_xlim(0, 12)
ax.set_ylim(0, 8)
ax.set_aspect('equal')
ax.axis('off')
ax.set_title('Modelo Hibrido-$\\pi$ del BJT (pequena senal)', fontsize=16, fontweight='bold')

# Nodos
B = (2, 4)
E = (6, 1.5)
C = (10, 4)
E_top = (6, 4)

# Base -> rpi -> Emisor
ax.annotate('', xy=(3.5, 4), xytext=(2.2, 4),
            arrowprops=dict(arrowstyle='-', lw=2, color=COLOR_PRINCIPAL))
ax.add_patch(FancyBboxPatch((3.5, 3.5), 1.8, 1.0, boxstyle='round,pad=0.1',
             facecolor=COLOR_CLARO, edgecolor=COLOR_PRINCIPAL, lw=2))
ax.text(4.4, 4.0, '$r_\\pi$', ha='center', va='center', fontsize=16, fontweight='bold',
        color=COLOR_PRINCIPAL)
ax.annotate('', xy=(6, 4), xytext=(5.3, 4),
            arrowprops=dict(arrowstyle='-', lw=2, color=COLOR_PRINCIPAL))

# rpi to E node vertical
ax.plot([6, 6], [4, 1.5], lw=2, color=COLOR_FIJO, ls='--')

# Fuente de corriente gm*vpi
ax.annotate('', xy=(10, 4), xytext=(10, 1.5),
            arrowprops=dict(arrowstyle='->', lw=2.5, color=COLOR_ROJO))
ax.add_patch(plt.Circle((10, 2.75), 0.5, fill=False, edgecolor=COLOR_ROJO, lw=2))
ax.text(10, 2.75, '$g_m v_\\pi$', ha='center', va='center', fontsize=13,
        fontweight='bold', color=COLOR_ROJO)

# ro en paralelo
ax.add_patch(FancyBboxPatch((7.5, 3.5), 1.5, 1.0, boxstyle='round,pad=0.1',
             facecolor='#fff5f0', edgecolor=COLOR_NARANJA, lw=2))
ax.text(8.25, 4.0, '$r_o$', ha='center', va='center', fontsize=16, fontweight='bold',
        color=COLOR_NARANJA)
ax.plot([6, 7.5], [4, 4], lw=2, color=COLOR_FIJO)
ax.plot([9, 10], [4, 4], lw=2, color=COLOR_FIJO)

# Linea inferior (emisor)
ax.plot([6, 10], [1.5, 1.5], lw=2, color=COLOR_FIJO)

# Etiquetas de nodos
ax.text(1.8, 4.0, 'B', ha='center', va='center', fontsize=18, fontweight='bold',
        color=COLOR_MORADO, bbox=dict(boxstyle='circle', facecolor='white', edgecolor=COLOR_MORADO, lw=2))
ax.text(10.5, 4.5, 'C', ha='center', va='center', fontsize=18, fontweight='bold',
        color=COLOR_MORADO, bbox=dict(boxstyle='circle', facecolor='white', edgecolor=COLOR_MORADO, lw=2))
ax.text(6, 0.9, 'E', ha='center', va='center', fontsize=18, fontweight='bold',
        color=COLOR_MORADO, bbox=dict(boxstyle='circle', facecolor='white', edgecolor=COLOR_MORADO, lw=2))

# vpi label
ax.annotate('', xy=(2.5, 3.0), xytext=(5.5, 3.0),
            arrowprops=dict(arrowstyle='<->', lw=1.5, color=COLOR_PUNTO))
ax.text(4.0, 2.5, '$v_\\pi$', ha='center', va='center', fontsize=15, fontweight='bold',
        color=COLOR_PUNTO)

plt.tight_layout()
plt.show()

---

## 2. Comparacion BJT vs MOS: Parametros de Pequena Senal

| Parametro | BJT | MOSFET |
|-----------|-----|--------|
| Transconductancia | $g_m = \dfrac{I_C}{V_T} \approx 3.846$ mA/V | $g_m = \dfrac{2I_D}{V_{EFF}}$ (menor para misma $I$) |
| Impedancia de entrada | $r_\pi = \dfrac{\beta}{g_m} \approx 1\,k\Omega$ (finita) | $r_{gs} \to \infty$ (infinita) |
| Impedancia de salida | $r_o = \dfrac{V_A}{I_C} \approx 20\,k\Omega$ | $r_o = \dfrac{1}{\lambda I_D}$ |
| Ganancia intrinseca | $g_m \cdot r_o = \dfrac{V_A}{V_T}$ | $g_m \cdot r_o = \dfrac{2}{\lambda V_{EFF}}$ |

> **Ventaja del BJT:** Mayor $g_m$ para la misma corriente ($V_T = 26$ mV vs $V_{EFF} \sim 200$ mV tipico).
>
> **Ventaja del MOS:** Impedancia de entrada infinita (no consume corriente de gate).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 5))

# Datos BJT vs MOS (para IC=ID=100uA)
gm_bjt = 3.846   # mA/V
gm_mos = 1.0     # mA/V (tipico con VEFF=200mV)
rin_bjt = 1.0    # kohm
rin_mos = 1000   # kohm (representamos como "muy alto")
ro_bjt = 20.0    # kohm
ro_mos = 20.0    # kohm (similar)

labels = ['BJT', 'MOS']
colors = [COLOR_PRINCIPAL, COLOR_NARANJA]

# gm
axes[0].bar(labels, [gm_bjt, gm_mos], color=colors, edgecolor='white', lw=2, width=0.5)
axes[0].set_ylabel('$g_m$ (mA/V)')
axes[0].set_title('Transconductancia $g_m$', fontweight='bold')
for i, v in enumerate([gm_bjt, gm_mos]):
    axes[0].text(i, v + 0.1, f'{v:.3f}', ha='center', fontweight='bold', fontsize=12)

# rin
axes[1].bar(labels, [rin_bjt, 50], color=colors, edgecolor='white', lw=2, width=0.5)
axes[1].set_ylabel('$r_{in}$ (k$\\Omega$)')
axes[1].set_title('Impedancia de Entrada $r_{in}$', fontweight='bold')
axes[1].text(0, rin_bjt + 1.5, f'{rin_bjt:.1f} k$\\Omega$', ha='center', fontweight='bold', fontsize=12)
axes[1].text(1, 50 + 1.5, '$\\infty$', ha='center', fontweight='bold', fontsize=14, color=COLOR_NARANJA)
axes[1].set_ylim(0, 65)

# ro
axes[2].bar(labels, [ro_bjt, ro_mos], color=colors, edgecolor='white', lw=2, width=0.5)
axes[2].set_ylabel('$r_o$ (k$\\Omega$)')
axes[2].set_title('Impedancia de Salida $r_o$', fontweight='bold')
for i, v in enumerate([ro_bjt, ro_mos]):
    axes[2].text(i, v + 0.5, f'{v:.1f}', ha='center', fontweight='bold', fontsize=12)

for ax in axes:
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.suptitle('Comparacion BJT vs MOS ($I_C = I_D = 100\\,\\mu A$)', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---

## 3. Amplificador Emisor Comun (CE)

Es la configuracion **fundamental** del BJT, equivalente al Source Comun (CS) en MOS.

El transistor amplifica la senal de entrada (base) y la invierte en la salida (colector).

### Esquema del circuito

- Entrada: senal $v_s$ con resistencia de fuente $R_S$
- Transistor: BJT en configuracion emisor comun
- Salida: en el nodo de colector, con $R_L$ (o carga activa)

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(12, 8))
ax.set_xlim(0, 14)
ax.set_ylim(0, 10)
ax.set_aspect('equal')
ax.axis('off')
ax.set_title('Circuito Emisor Comun (CE) - Pequena Senal', fontsize=16, fontweight='bold')

# Fuente vs
ax.text(1.0, 5.0, '$v_s$', ha='center', va='center', fontsize=16, fontweight='bold',
        color=COLOR_PUNTO, bbox=dict(boxstyle='circle,pad=0.3', facecolor='#e5f5e0', edgecolor=COLOR_PUNTO, lw=2))
ax.annotate('', xy=(2.5, 5.0), xytext=(1.5, 5.0),
            arrowprops=dict(arrowstyle='-', lw=2, color=COLOR_FIJO))

# RS
ax.add_patch(FancyBboxPatch((2.5, 4.5), 2.0, 1.0, boxstyle='round,pad=0.1',
             facecolor=COLOR_CLARO, edgecolor=COLOR_PRINCIPAL, lw=2))
ax.text(3.5, 5.0, '$R_S$', ha='center', va='center', fontsize=15, fontweight='bold',
        color=COLOR_PRINCIPAL)

# Wire RS to Base
ax.plot([4.5, 6], [5, 5], lw=2, color=COLOR_FIJO)

# BJT Triangle
triangle = plt.Polygon([(6, 3), (6, 7), (8, 5)], fill=True, facecolor='#deebf7',
                        edgecolor=COLOR_PRINCIPAL, lw=2.5)
ax.add_patch(triangle)
ax.text(6.5, 5.0, 'BJT', ha='center', va='center', fontsize=13, fontweight='bold',
        color=COLOR_PRINCIPAL)

# Labels B, C, E
ax.text(5.5, 5.0, 'B', ha='center', va='center', fontsize=14, fontweight='bold',
        color=COLOR_MORADO)
ax.text(8.5, 6.0, 'C', ha='center', va='center', fontsize=14, fontweight='bold',
        color=COLOR_MORADO)
ax.text(7.0, 2.5, 'E', ha='center', va='center', fontsize=14, fontweight='bold',
        color=COLOR_MORADO)

# Colector up to RL
ax.plot([8, 8], [5.5, 6.5], lw=2, color=COLOR_FIJO)
ax.plot([8, 10], [6.5, 6.5], lw=2, color=COLOR_FIJO)

# RL
ax.add_patch(FancyBboxPatch((10, 6.0), 2.0, 1.0, boxstyle='round,pad=0.1',
             facecolor='#fff5f0', edgecolor=COLOR_ROJO, lw=2))
ax.text(11.0, 6.5, '$R_L$', ha='center', va='center', fontsize=15, fontweight='bold',
        color=COLOR_ROJO)

# RL to ground
ax.plot([12, 13], [6.5, 6.5], lw=2, color=COLOR_FIJO)
ax.plot([13, 13], [6.5, 2], lw=2, color=COLOR_FIJO)

# Emisor to ground
ax.plot([7, 7], [3, 2], lw=2, color=COLOR_FIJO)

# Ground line
ax.plot([1, 13], [2, 2], lw=2, color=COLOR_FIJO)
ax.plot([1, 1], [2, 5], lw=2, color=COLOR_FIJO)

# Ground symbol
for offset in [0.0, 0.15, 0.3]:
    w = 1.0 - offset*2
    ax.plot([7 - w/2, 7 + w/2], [1.7 - offset, 1.7 - offset], lw=2, color=COLOR_FIJO)

# Vout label
ax.annotate('$v_{out}$', xy=(9, 6.5), fontsize=16, fontweight='bold', color=COLOR_ROJO,
            bbox=dict(boxstyle='round,pad=0.2', facecolor='white', edgecolor=COLOR_ROJO, lw=1.5))

# ro indicator (dashed)
ax.plot([8.5, 8.5], [3.5, 5.5], lw=1.5, ls=':', color=COLOR_NARANJA)
ax.text(9.2, 4.5, '$r_o$', fontsize=14, color=COLOR_NARANJA, fontweight='bold')

plt.tight_layout()
plt.show()

### Analisis del Emisor Comun (CE)

Aplicando el modelo hibrido-$\pi$ al circuito CE:

$$\boxed{A_v = -\frac{r_\pi}{r_\pi + R_S} \cdot g_m \cdot (r_o \| R_L)}$$

$$\boxed{r_{in} = r_\pi}$$

$$\boxed{r_{out} = r_o}$$

Donde $r_o \| R_L = \frac{r_o \cdot R_L}{r_o + R_L}$

**Nota:** El signo negativo en $A_v$ indica **inversion de fase** (igual que en CS con MOS).

In [ ]:
# === Emisor Comun (CE): Calculo numerico ===
print('=== EMISOR COMUN (CE) ===')
print(f'Parametros: gm1={gm1*1e3:.3f} mA/V, rpi1={rpi1:.0f} ohm, ro1={ro1:.0f} ohm')
print(f'            RS={RS:.0f} ohm, RL={RL:.0f} ohm')
print()

# Resistencia de entrada
rin_ce = rpi1
print(f'rin = rpi1 = {rin_ce:.0f} ohm = {rin_ce*1e-3:.3f} kohm')

# Carga efectiva
ro_par_RL = (ro1 * RL) / (ro1 + RL)
print(f'ro1 || RL = {ro_par_RL:.1f} ohm = {ro_par_RL*1e-3:.3f} kohm')

# Ganancia de voltaje
divisor_entrada = rpi1 / (rpi1 + RS)
Av_ce = -divisor_entrada * gm1 * ro_par_RL
print(f'Divisor entrada: rpi/(rpi+RS) = {divisor_entrada:.4f}')
print(f'Av = -{divisor_entrada:.4f} * {gm1*1e3:.3f}e-3 * {ro_par_RL:.1f}')
print(f'Av = {Av_ce:.3f}')

# Resistencia de salida
rout_ce = ro1
print(f'rout = ro1 = {rout_ce:.0f} ohm = {rout_ce*1e-3:.1f} kohm')

---

## 4. Amplificador Colector Comun (CC) - Seguidor de Emisor

Es la configuracion **buffer** del BJT, equivalente al Drain Comun (CD) en MOS.

La salida se toma del emisor. El colector va directamente a $V_{CC}$ (AC ground).

### Caracteristicas principales:
- Ganancia de voltaje $A_v \approx 1$ (pero ligeramente menor)
- **Alta** impedancia de entrada
- **Baja** impedancia de salida
- **No invierte** la senal

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(12, 8))
ax.set_xlim(0, 14)
ax.set_ylim(0, 10)
ax.set_aspect('equal')
ax.axis('off')
ax.set_title('Circuito Colector Comun (CC) - Seguidor de Emisor', fontsize=16, fontweight='bold')

# Fuente vs
ax.text(1.0, 6.0, '$v_s$', ha='center', va='center', fontsize=16, fontweight='bold',
        color=COLOR_PUNTO, bbox=dict(boxstyle='circle,pad=0.3', facecolor='#e5f5e0', edgecolor=COLOR_PUNTO, lw=2))
ax.annotate('', xy=(2.5, 6.0), xytext=(1.5, 6.0),
            arrowprops=dict(arrowstyle='-', lw=2, color=COLOR_FIJO))

# RS
ax.add_patch(FancyBboxPatch((2.5, 5.5), 2.0, 1.0, boxstyle='round,pad=0.1',
             facecolor=COLOR_CLARO, edgecolor=COLOR_PRINCIPAL, lw=2))
ax.text(3.5, 6.0, '$R_S$', ha='center', va='center', fontsize=15, fontweight='bold',
        color=COLOR_PRINCIPAL)

# Wire RS to Base
ax.plot([4.5, 6], [6, 6], lw=2, color=COLOR_FIJO)

# BJT Triangle
triangle = plt.Polygon([(6, 4), (6, 8), (8, 6)], fill=True, facecolor='#deebf7',
                        edgecolor=COLOR_PRINCIPAL, lw=2.5)
ax.add_patch(triangle)
ax.text(6.5, 6.0, 'BJT', ha='center', va='center', fontsize=13, fontweight='bold',
        color=COLOR_PRINCIPAL)

# Labels B, C, E
ax.text(5.5, 6.0, 'B', ha='center', va='center', fontsize=14, fontweight='bold',
        color=COLOR_MORADO)
ax.text(7.5, 8.5, 'C', ha='center', va='center', fontsize=14, fontweight='bold',
        color=COLOR_MORADO)
ax.text(8.5, 4.5, 'E', ha='center', va='center', fontsize=14, fontweight='bold',
        color=COLOR_MORADO)

# Colector up to VCC (AC ground)
ax.plot([7, 7], [8, 9], lw=2, color=COLOR_FIJO)
ax.text(7.0, 9.3, '$V_{CC}$', ha='center', va='center', fontsize=14, fontweight='bold',
        color=COLOR_MORADO)
ax.text(7.0, 9.7, '(AC gnd)', ha='center', va='center', fontsize=10, color=COLOR_FIJO)

# Emisor down to RL and output
ax.plot([8, 8], [4.5, 3.5], lw=2, color=COLOR_FIJO)
ax.plot([8, 10], [3.5, 3.5], lw=2, color=COLOR_FIJO)

# RL
ax.add_patch(FancyBboxPatch((10, 3.0), 2.0, 1.0, boxstyle='round,pad=0.1',
             facecolor='#fff5f0', edgecolor=COLOR_ROJO, lw=2))
ax.text(11.0, 3.5, '$R_L$', ha='center', va='center', fontsize=15, fontweight='bold',
        color=COLOR_ROJO)

# RL to ground
ax.plot([12, 13], [3.5, 3.5], lw=2, color=COLOR_FIJO)
ax.plot([13, 13], [3.5, 2], lw=2, color=COLOR_FIJO)

# Vout label
ax.annotate('$v_{out}$', xy=(9, 3.5), fontsize=16, fontweight='bold', color=COLOR_ROJO,
            bbox=dict(boxstyle='round,pad=0.2', facecolor='white', edgecolor=COLOR_ROJO, lw=1.5))

# Ground
ax.plot([1, 13], [2, 2], lw=2, color=COLOR_FIJO)
ax.plot([1, 1], [2, 6], lw=2, color=COLOR_FIJO)
for offset in [0.0, 0.15, 0.3]:
    w = 1.0 - offset*2
    ax.plot([7 - w/2, 7 + w/2], [1.7 - offset, 1.7 - offset], lw=2, color=COLOR_FIJO)

# ro indicator
ax.plot([8.5, 8.5], [5.0, 7.0], lw=1.5, ls=':', color=COLOR_NARANJA)
ax.text(9.2, 6.0, '$r_o$', fontsize=14, color=COLOR_NARANJA, fontweight='bold')

plt.tight_layout()
plt.show()

### Analisis del Colector Comun (CC)

Aplicando el modelo hibrido-$\pi$ al circuito CC:

$$\boxed{A_v \approx \frac{1}{1 + \dfrac{R_S + r_\pi}{(1 + g_m r_\pi)(r_o \| R_L)}}}$$

$$\boxed{r_{in} = r_\pi + (1 + g_m r_\pi)(r_o \| R_L)}$$

$$\boxed{r_{out} = \frac{R_S + r_\pi}{1 + g_m r_\pi}}$$

**Observacion:** La ganancia es cercana a 1, pero la $r_\pi$ finita del BJT reduce
la impedancia de entrada respecto al caso MOS (donde $r_{in} \to \infty$).

In [ ]:
# === Colector Comun (CC): Calculo numerico ===
print('=== COLECTOR COMUN (CC) - Seguidor de Emisor ===')
print(f'Parametros: gm1={gm1*1e3:.3f} mA/V, rpi1={rpi1:.0f} ohm, ro1={ro1:.0f} ohm')
print(f'            RS={RS:.0f} ohm, RL={RL:.0f} ohm')
print()

# Resistencia de entrada
rin_cc = rpi1 + (1 + gm1 * rpi1) * ro_par_RL
print(f'rin = rpi + (1 + gm*rpi)*(ro||RL)')
print(f'    = {rpi1:.0f} + (1 + {gm1*rpi1:.1f}) * {ro_par_RL:.1f}')
print(f'    = {rin_cc:.1f} ohm = {rin_cc*1e-3:.3f} kohm')
print()

# Ganancia de voltaje
numerator_cc = 1.0
denominator_cc = 1 + (RS + rpi1) / ((1 + gm1 * rpi1) * ro_par_RL)
Av_cc = numerator_cc / denominator_cc
print(f'Av = 1 / (1 + (RS+rpi)/((1+gm*rpi)*(ro||RL)))')
print(f'   = 1 / (1 + ({RS+rpi1:.0f}) / ({(1+gm1*rpi1):.1f} * {ro_par_RL:.1f}))')
print(f'   = {Av_cc:.4f}')
print()

# Resistencia de salida
rout_cc = (RS + rpi1) / (1 + gm1 * rpi1)
print(f'rout = (RS + rpi) / (1 + gm*rpi)')
print(f'     = ({RS+rpi1:.0f}) / ({1+gm1*rpi1:.1f})')
print(f'     = {rout_cc:.1f} ohm = {rout_cc*1e-3:.3f} kohm')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

configs = ['CE', 'CC']
colors_config = [COLOR_PRINCIPAL, COLOR_ROJO]

# Av
av_vals = [abs(Av_ce), Av_cc]
axes[0].bar(configs, av_vals, color=colors_config, edgecolor='white', lw=2, width=0.5)
axes[0].set_ylabel('$|A_v|$')
axes[0].set_title('Ganancia de Voltaje $|A_v|$', fontweight='bold')
for i, v in enumerate(av_vals):
    axes[0].text(i, v + 0.3, f'{v:.3f}', ha='center', fontweight='bold', fontsize=13)
axes[0].text(0, -1.5, '(invierte)', ha='center', fontsize=10, color=COLOR_FIJO, style='italic')
axes[0].text(1, -1.5, '(no invierte)', ha='center', fontsize=10, color=COLOR_FIJO, style='italic')

# rin
rin_vals = [rin_ce * 1e-3, rin_cc * 1e-3]
axes[1].bar(configs, rin_vals, color=colors_config, edgecolor='white', lw=2, width=0.5)
axes[1].set_ylabel('$r_{in}$ (k$\\Omega$)')
axes[1].set_title('Impedancia de Entrada $r_{in}$', fontweight='bold')
for i, v in enumerate(rin_vals):
    axes[1].text(i, v + 0.3, f'{v:.3f}', ha='center', fontweight='bold', fontsize=13)

# rout
rout_vals = [rout_ce * 1e-3, rout_cc * 1e-3]
axes[2].bar(configs, rout_vals, color=colors_config, edgecolor='white', lw=2, width=0.5)
axes[2].set_ylabel('$r_{out}$ (k$\\Omega$)')
axes[2].set_title('Impedancia de Salida $r_{out}$', fontweight='bold')
for i, v in enumerate(rout_vals):
    axes[2].text(i, v + 0.5, f'{v:.3f}', ha='center', fontweight='bold', fontsize=13)

for ax in axes:
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.suptitle('Comparacion CE vs CC (BJT)', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---

## 5. Cargas Activas: Espejo de Corriente como Carga

Sustituir $R_L$ por un **espejo de corriente** (carga activa) aumenta drasticamente la ganancia.

Con carga activa, $R_L \to r_{o,carga}$ (resistencia de salida del espejo), que es mucho mayor.

$$A_v^{\text{activa}} = -g_m \cdot (r_{o1} \| r_{o,carga})$$

Si $r_{o,carga} \approx r_{o1}$:

$$\boxed{A_v^{\text{activa}} \approx -\frac{g_m \cdot r_o}{2}}$$

> Con carga activa, la ganancia se acerca al limite intrinseco $g_m \cdot r_o$,
> que para el BJT es $V_A / V_T \approx 77$ (con $V_A = 2$ V).

In [ ]:
# === Carga Activa: Espejo de Corriente ===
print('=== CE CON CARGA ACTIVA (Espejo de Corriente) ===')
print()

ro_carga = ro1  # Asumimos espejo simetrico
ro_par_activa = (ro1 * ro_carga) / (ro1 + ro_carga)

Av_activa = -gm1 * ro_par_activa
gm_ro = gm1 * ro1

print(f'ro_carga = ro1 = {ro_carga:.0f} ohm')
print(f'ro1 || ro_carga = {ro_par_activa:.0f} ohm = {ro_par_activa*1e-3:.1f} kohm')
print(f'Av(activa) = -gm * (ro||ro_carga) = {Av_activa:.2f}')
print()
print(f'Ganancia intrinseca gm*ro = {gm_ro:.2f}')
print(f'VA/Vt = {VA/Vt:.1f}')
print()
print(f'Mejora respecto a carga pasiva (RL={RL*1e-3:.0f}k):')
print(f'  |Av(pasiva)|  = {abs(Av_ce):.2f}')
print(f'  |Av(activa)|  = {abs(Av_activa):.2f}')
print(f'  Factor mejora = {abs(Av_activa)/abs(Av_ce):.1f}x')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

labels_carga = ['CE + $R_L$ pasiva\n($R_L = 5\\,k\\Omega$)',
                'CE + Carga Activa\n(espejo de corriente)']
vals_carga = [abs(Av_ce), abs(Av_activa)]
colors_carga = [COLOR_PRINCIPAL, COLOR_PUNTO]

bars = ax.bar(labels_carga, vals_carga, color=colors_carga, edgecolor='white', lw=2, width=0.5)

for bar, v in zip(bars, vals_carga):
    ax.text(bar.get_x() + bar.get_width()/2, v + 0.8,
            f'$|A_v| = {v:.1f}$', ha='center', fontweight='bold', fontsize=14)

ax.set_ylabel('$|A_v|$', fontsize=14)
ax.set_title('Efecto de la Carga Activa en la Ganancia CE', fontsize=15, fontweight='bold')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# Flecha de mejora
ax.annotate(f'{abs(Av_activa)/abs(Av_ce):.1f}x', xy=(1, abs(Av_activa)/2),
            fontsize=20, fontweight='bold', color=COLOR_ROJO, ha='center')

plt.tight_layout()
plt.show()

---

## 6. Combinacion de Etapas: Cascode (CE + CB)

El **cascode** combina un Emisor Comun (CE) con un Base Comun (CB) en serie.
Es equivalente al CS + CG en tecnologia MOS.

### Objetivo: Aumentar $r_{out}$

La etapa CB "mira" hacia el emisor de Q2 y ve la $r_o$ del CE amplificada:

$$\boxed{r_{out}^{\text{cascode}} \approx g_{m2} \cdot r_{o2} \cdot r_{o1}}$$

Esto resulta en una resistencia de salida **mucho mayor** que un CE simple.

### Ganancia del Cascode

Con carga activa cascode:

$$A_v^{\text{cascode}} \approx -g_m \cdot (r_{out,\text{cascode}} \| r_{out,\text{carga\_cascode}})$$

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(10, 10))
ax.set_xlim(0, 12)
ax.set_ylim(0, 12)
ax.set_aspect('equal')
ax.axis('off')
ax.set_title('Cascode BJT (CE + CB)', fontsize=16, fontweight='bold')

# Q1 - Emisor Comun (bottom)
q1_x, q1_y = 5, 3
tri1 = plt.Polygon([(q1_x-1, q1_y-1.5), (q1_x-1, q1_y+1.5), (q1_x+1, q1_y)],
                    fill=True, facecolor='#deebf7', edgecolor=COLOR_PRINCIPAL, lw=2.5)
ax.add_patch(tri1)
ax.text(q1_x-0.5, q1_y, '$Q_1$\nCE', ha='center', va='center', fontsize=12,
        fontweight='bold', color=COLOR_PRINCIPAL)

# Q2 - Base Comun (top)
q2_x, q2_y = 5, 7
tri2 = plt.Polygon([(q2_x-1, q2_y-1.5), (q2_x-1, q2_y+1.5), (q2_x+1, q2_y)],
                    fill=True, facecolor='#e5f5e0', edgecolor=COLOR_PUNTO, lw=2.5)
ax.add_patch(tri2)
ax.text(q2_x-0.5, q2_y, '$Q_2$\nCB', ha='center', va='center', fontsize=12,
        fontweight='bold', color=COLOR_PUNTO)

# Connection Q1 colector -> Q2 emisor
ax.plot([q1_x+0.5, q1_x+0.5], [q1_y+0.8, q2_y-1.5], lw=2.5, color=COLOR_ROJO)
ax.text(q1_x+1.2, 5.0, 'Nodo\ninterno', ha='center', va='center', fontsize=10,
        color=COLOR_ROJO, style='italic')

# Input to Q1 base
ax.plot([1, q1_x-1], [3, 3], lw=2, color=COLOR_FIJO)
ax.text(0.5, 3.0, '$v_{in}$', ha='center', va='center', fontsize=15, fontweight='bold',
        color=COLOR_PUNTO, bbox=dict(boxstyle='circle,pad=0.3', facecolor='#e5f5e0',
        edgecolor=COLOR_PUNTO, lw=2))

# Q2 base to VBIAS
ax.plot([q2_x-1, 2], [7, 7], lw=2, color=COLOR_FIJO)
ax.text(1.5, 7.0, '$V_{BIAS}$', ha='center', va='center', fontsize=13, fontweight='bold',
        color=COLOR_MORADO, bbox=dict(boxstyle='round,pad=0.2', facecolor='white',
        edgecolor=COLOR_MORADO, lw=1.5))

# Q1 emisor to ground
ax.plot([q1_x-0.5, q1_x-0.5], [q1_y-1.5, 0.5], lw=2, color=COLOR_FIJO)
for offset in [0.0, 0.15, 0.3]:
    w = 0.8 - offset*2
    ax.plot([q1_x-0.5-w/2, q1_x-0.5+w/2], [0.3-offset, 0.3-offset], lw=2, color=COLOR_FIJO)

# Q2 colector to output
ax.plot([q2_x+0.5, q2_x+0.5], [q2_y+0.8, 10], lw=2, color=COLOR_FIJO)
ax.plot([q2_x+0.5, 8], [10, 10], lw=2, color=COLOR_FIJO)
ax.text(8.5, 10.0, '$v_{out}$', ha='center', va='center', fontsize=15, fontweight='bold',
        color=COLOR_ROJO, bbox=dict(boxstyle='round,pad=0.2', facecolor='white',
        edgecolor=COLOR_ROJO, lw=1.5))

# rout annotation
ax.annotate('$r_{out} \\approx g_m r_o^2$', xy=(8, 8.5), fontsize=15, fontweight='bold',
            color=COLOR_NARANJA, bbox=dict(boxstyle='round,pad=0.3', facecolor='#fff5f0',
            edgecolor=COLOR_NARANJA, lw=2))

plt.tight_layout()
plt.show()

In [ ]:
# === CASCODE (CE + CB) ===
print('=== CASCODE (CE + CB) ===')
print()

# rout del cascode
rout_cascode = gm1 * ro1 * ro1  # gm * ro * ro
print(f'rout_cascode = gm1 * ro1 * ro1')
print(f'             = {gm1*1e3:.3f}e-3 * {ro1:.0f} * {ro1:.0f}')
print(f'             = {rout_cascode:.0f} ohm = {rout_cascode*1e-6:.3f} Mohm')
print()

# Comparacion con CE simple
print(f'Comparacion rout:')
print(f'  CE simple: rout = {rout_ce:.0f} ohm = {rout_ce*1e-3:.1f} kohm')
print(f'  Cascode:   rout = {rout_cascode:.0f} ohm = {rout_cascode*1e-6:.3f} Mohm')
print(f'  Factor mejora = gm*ro = {gm1*ro1:.1f}x')
print()

# Ganancia con carga activa cascode
rout_carga_cascode = rout_cascode  # Asumimos carga cascode simetrica
R_eff_cascode = (rout_cascode * rout_carga_cascode) / (rout_cascode + rout_carga_cascode)
Av_cascode = -gm1 * R_eff_cascode
print(f'Con carga activa cascode:')
print(f'Av_cascode = -gm * (rout_casc || rout_carga_casc) = {Av_cascode:.0f}')

---

## 7. Combinacion de Etapas: Buffer (CE + CC)

El **buffer** combina un Emisor Comun (CE) para ganancia con un Colector Comun (CC)
para reducir la impedancia de salida.

Es equivalente al CS + CD en tecnologia MOS.

### Ventajas
- Ganancia de voltaje elevada (proporcionada por CE)
- Impedancia de salida baja (proporcionada por CC)
- Ideal para **driving low-impedance loads**

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(14, 7))
ax.set_xlim(0, 16)
ax.set_ylim(0, 10)
ax.set_aspect('equal')
ax.axis('off')
ax.set_title('Buffer BJT: CE + CC', fontsize=16, fontweight='bold')

# Etapa 1: CE
box1_x, box1_y = 3, 4
ax.add_patch(FancyBboxPatch((box1_x-1.5, box1_y-1.5), 3, 3,
             boxstyle='round,pad=0.2', facecolor='#deebf7', edgecolor=COLOR_PRINCIPAL, lw=3))
ax.text(box1_x, box1_y+0.5, 'Etapa 1', ha='center', va='center', fontsize=14,
        fontweight='bold', color=COLOR_PRINCIPAL)
ax.text(box1_x, box1_y-0.3, 'CE', ha='center', va='center', fontsize=20,
        fontweight='bold', color=COLOR_PRINCIPAL)
ax.text(box1_x, box1_y-1.0, '$A_v$ alto', ha='center', va='center', fontsize=11,
        color=COLOR_FIJO, style='italic')

# Flecha de conexion
ax.annotate('', xy=(7.5, 4), xytext=(4.7, 4),
            arrowprops=dict(arrowstyle='->', lw=3, color=COLOR_ROJO))

# Etapa 2: CC
box2_x, box2_y = 9, 4
ax.add_patch(FancyBboxPatch((box2_x-1.5, box2_y-1.5), 3, 3,
             boxstyle='round,pad=0.2', facecolor='#e5f5e0', edgecolor=COLOR_PUNTO, lw=3))
ax.text(box2_x, box2_y+0.5, 'Etapa 2', ha='center', va='center', fontsize=14,
        fontweight='bold', color=COLOR_PUNTO)
ax.text(box2_x, box2_y-0.3, 'CC', ha='center', va='center', fontsize=20,
        fontweight='bold', color=COLOR_PUNTO)
ax.text(box2_x, box2_y-1.0, '$r_{out}$ bajo', ha='center', va='center', fontsize=11,
        color=COLOR_FIJO, style='italic')

# Input
ax.annotate('', xy=(1.5, 4), xytext=(0.5, 4),
            arrowprops=dict(arrowstyle='->', lw=2.5, color=COLOR_FIJO))
ax.text(0.3, 4.0, '$v_{in}$', ha='center', va='center', fontsize=15, fontweight='bold',
        color=COLOR_PUNTO)

# Output
ax.annotate('', xy=(13, 4), xytext=(10.5, 4),
            arrowprops=dict(arrowstyle='->', lw=2.5, color=COLOR_FIJO))
ax.text(13.5, 4.0, '$v_{out}$', ha='center', va='center', fontsize=15, fontweight='bold',
        color=COLOR_ROJO)

# Properties boxes
ax.add_patch(FancyBboxPatch((0.5, 7), 5, 1.5, boxstyle='round,pad=0.15',
             facecolor='white', edgecolor=COLOR_PRINCIPAL, lw=1.5, ls='--'))
ax.text(3.0, 7.75, '$A_v$ alto  |  $r_{out}$ alto', ha='center', va='center',
        fontsize=12, color=COLOR_PRINCIPAL)

ax.add_patch(FancyBboxPatch((7, 7), 5, 1.5, boxstyle='round,pad=0.15',
             facecolor='white', edgecolor=COLOR_PUNTO, lw=1.5, ls='--'))
ax.text(9.5, 7.75, '$A_v \\approx 1$  |  $r_{out}$ bajo', ha='center', va='center',
        fontsize=12, color=COLOR_PUNTO)

# Result
ax.add_patch(FancyBboxPatch((3.5, 0.5), 8, 1.5, boxstyle='round,pad=0.2',
             facecolor='#fff5f0', edgecolor=COLOR_ROJO, lw=2.5))
ax.text(7.5, 1.25, 'Resultado: $A_v$ alto + $r_{out}$ bajo', ha='center', va='center',
        fontsize=14, fontweight='bold', color=COLOR_ROJO)

plt.tight_layout()
plt.show()

In [ ]:
# === BUFFER (CE + CC) ===
print('=== BUFFER (CE + CC) ===')
print()

# Etapa CE: ganancia y rout
print('Etapa 1 (CE):')
print(f'  Av1 = {Av_ce:.3f}')
print(f'  rout1 = ro1 = {rout_ce*1e-3:.1f} kohm')
print()

# Etapa CC ve RS_eff = rout_CE
RS_eff = rout_ce  # La CC ve la rout del CE como su RS
rin_cc_2 = rpi1 + (1 + gm1 * rpi1) * ro_par_RL
Av_cc_2 = 1 / (1 + (RS_eff + rpi1) / ((1 + gm1 * rpi1) * ro_par_RL))
rout_cc_2 = (RS_eff + rpi1) / (1 + gm1 * rpi1)

print('Etapa 2 (CC) con RS_eff = rout_CE:')
print(f'  RS_eff = {RS_eff*1e-3:.1f} kohm')
print(f'  Av2 = {Av_cc_2:.4f}')
print(f'  rout_buffer = {rout_cc_2:.1f} ohm = {rout_cc_2*1e-3:.3f} kohm')
print()

# Ganancia total
Av_buffer = Av_ce * Av_cc_2
print(f'Ganancia total:')
print(f'  Av_total = Av1 * Av2 = {Av_ce:.3f} * {Av_cc_2:.4f} = {Av_buffer:.3f}')
print()

print(f'Comparacion rout:')
print(f'  CE solo:  rout = {rout_ce*1e-3:.1f} kohm')
print(f'  Buffer:   rout = {rout_cc_2*1e-3:.3f} kohm')
print(f'  Reduccion = {rout_ce/rout_cc_2:.1f}x')

---

## 8. Tabla Resumen: Todas las Configuraciones

Valores calculados con: $I_C = 100\,\mu A$, $V_T = 26$ mV, $\beta = 100$, $V_A = 2$ V, $R_S = 100\,\Omega$, $R_L = 5\,k\Omega$

In [ ]:
fig, ax = plt.subplots(figsize=(14, 8))
ax.axis('off')

# Table data
col_labels = ['Config.', '$A_v$', '$r_{in}$', '$r_{out}$', 'Aplicacion']
row_data = [
    ['CE\n(Emisor Comun)', f'{Av_ce:.2f}', f'{rin_ce*1e-3:.3f} k$\\Omega$',
     f'{rout_ce*1e-3:.1f} k$\\Omega$', 'Amplificacion\n(invierte)'],
    ['CC\n(Colector Comun)', f'{Av_cc:.3f}', f'{rin_cc*1e-3:.2f} k$\\Omega$',
     f'{rout_cc:.0f} $\\Omega$', 'Buffer\n(no invierte)'],
    ['CE + Activa', f'{Av_activa:.1f}', f'{rin_ce*1e-3:.3f} k$\\Omega$',
     f'{(ro_par_activa)*1e-3:.1f} k$\\Omega$', 'Alta ganancia'],
    ['Cascode\n(CE+CB)', f'{Av_cascode:.0f}', f'{rin_ce*1e-3:.3f} k$\\Omega$',
     f'{rout_cascode*1e-3:.0f} k$\\Omega$', 'Muy alta ganancia\n+ alto rout'],
    ['Buffer\n(CE+CC)', f'{Av_buffer:.2f}', f'{rin_ce*1e-3:.3f} k$\\Omega$',
     f'{rout_cc_2:.0f} $\\Omega$', 'Ganancia\n+ bajo rout'],
]

table = ax.table(cellText=row_data, colLabels=col_labels,
                  cellLoc='center', loc='center',
                  colWidths=[0.18, 0.15, 0.18, 0.18, 0.22])

table.auto_set_font_size(False)
table.set_fontsize(12)
table.scale(1.0, 2.5)

# Style header
for j in range(len(col_labels)):
    cell = table[0, j]
    cell.set_facecolor(COLOR_PRINCIPAL)
    cell.set_text_props(color='white', fontweight='bold', fontsize=13)

# Color rows
row_colors = ['#deebf7', '#e5f5e0', '#fff5f0', '#f0e6ff', '#fef0d9']
for i in range(len(row_data)):
    for j in range(len(col_labels)):
        table[i+1, j].set_facecolor(row_colors[i])
        table[i+1, j].set_edgecolor(COLOR_FIJO)

ax.set_title('Resumen de Configuraciones BJT', fontsize=18, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

---

## 9. Ejercicios Resueltos

### Ejercicio 1: Analisis CE con valores dados

**Enunciado:** Un BJT con $I_C = 200\,\mu A$, $\beta = 150$, $V_A = 5$ V se usa en
configuracion CE con $R_S = 50\,\Omega$ y $R_L = 10\,k\Omega$.
Calcular $g_m$, $r_\pi$, $r_o$, $A_v$, $r_{in}$, $r_{out}$.

In [ ]:
# === EJERCICIO 1: CE con valores dados ===
print('=== EJERCICIO 1: Emisor Comun ===')
print()

# Datos
IC_1 = 200e-6
beta_1 = 150
VA_1 = 5.0
RS_1 = 50
RL_1 = 10e3

# Paso 1: Parametros pequena senal
gm_1 = IC_1 / Vt
rpi_1 = beta_1 / gm_1
ro_1 = VA_1 / IC_1

print('Paso 1: Parametros de pequena senal')
print(f'  gm  = IC/Vt = {IC_1*1e6:.0f}u / {Vt*1e3:.0f}m = {gm_1*1e3:.3f} mA/V')
print(f'  rpi = beta/gm = {beta_1} / {gm_1*1e3:.3f}m = {rpi_1:.0f} ohm = {rpi_1*1e-3:.3f} kohm')
print(f'  ro  = VA/IC = {VA_1} / {IC_1*1e6:.0f}u = {ro_1:.0f} ohm = {ro_1*1e-3:.1f} kohm')
print()

# Paso 2: CE
ro_par_RL_1 = (ro_1 * RL_1) / (ro_1 + RL_1)
Av_1 = -(rpi_1 / (rpi_1 + RS_1)) * gm_1 * ro_par_RL_1
rin_1 = rpi_1
rout_1 = ro_1

print('Paso 2: Analisis CE')
print(f'  ro || RL = {ro_par_RL_1:.1f} ohm = {ro_par_RL_1*1e-3:.3f} kohm')
print(f'  Av = -(rpi/(rpi+RS)) * gm * (ro||RL)')
print(f'     = -({rpi_1:.0f}/({rpi_1:.0f}+{RS_1:.0f})) * {gm_1*1e3:.3f}e-3 * {ro_par_RL_1:.1f}')
print(f'     = {Av_1:.3f}')
print(f'  rin  = {rin_1:.0f} ohm = {rin_1*1e-3:.3f} kohm')
print(f'  rout = {rout_1:.0f} ohm = {rout_1*1e-3:.1f} kohm')

### Ejercicio 2: Analisis CC (Seguidor de Emisor)

**Enunciado:** Con los mismos parametros del Ejercicio 1, configurar el BJT como
colector comun. Calcular $A_v$, $r_{in}$, $r_{out}$ y verificar que actua como buffer.

In [ ]:
# === EJERCICIO 2: CC con mismos parametros ===
print('=== EJERCICIO 2: Colector Comun (Buffer) ===')
print()
print(f'Usando: gm={gm_1*1e3:.3f} mA/V, rpi={rpi_1:.0f} ohm, ro={ro_1:.0f} ohm')
print(f'        RS={RS_1:.0f} ohm, RL={RL_1:.0f} ohm')
print()

# Calculos CC
ro_par_RL_1cc = (ro_1 * RL_1) / (ro_1 + RL_1)
rin_2 = rpi_1 + (1 + gm_1 * rpi_1) * ro_par_RL_1cc
Av_2 = 1 / (1 + (RS_1 + rpi_1) / ((1 + gm_1 * rpi_1) * ro_par_RL_1cc))
rout_2 = (RS_1 + rpi_1) / (1 + gm_1 * rpi_1)

print('Resultados CC:')
print(f'  ro || RL = {ro_par_RL_1cc:.1f} ohm = {ro_par_RL_1cc*1e-3:.3f} kohm')
print(f'  rin = rpi + (1+gm*rpi)*(ro||RL)')
print(f'      = {rpi_1:.0f} + (1+{gm_1*rpi_1:.1f})*{ro_par_RL_1cc:.1f}')
print(f'      = {rin_2:.0f} ohm = {rin_2*1e-3:.2f} kohm')
print(f'  Av = {Av_2:.4f} (muy cercano a 1, confirma buffer)')
print(f'  rout = (RS+rpi)/(1+gm*rpi)')
print(f'       = ({RS_1+rpi_1:.0f})/({1+gm_1*rpi_1:.1f})')
print(f'       = {rout_2:.2f} ohm (muy bajo, confirma buffer)')

### Ejercicio 3: Cascode BJT

**Enunciado:** Dos transistores identicos ($I_C = 100\,\mu A$, $\beta = 100$, $V_A = 2$ V)
forman un cascode. Calcular $r_{out}$ del cascode y comparar con CE simple.

In [ ]:
# === EJERCICIO 3: Cascode ===
print('=== EJERCICIO 3: Cascode BJT ===')
print()

# Parametros (mismos Q1=Q2)
print(f'Q1 = Q2: IC={IC*1e6:.0f} uA, beta={beta}, VA={VA} V')
print(f'gm = {gm1*1e3:.3f} mA/V, rpi = {rpi1:.0f} ohm, ro = {ro1:.0f} ohm')
print()

# rout cascode
rout_casc = gm1 * ro1 * ro1
print(f'rout_cascode = gm * ro * ro')
print(f'             = {gm1*1e3:.3f}e-3 * {ro1:.0f} * {ro1:.0f}')
print(f'             = {rout_casc:.0f} ohm')
print(f'             = {rout_casc*1e-3:.1f} kohm')
print(f'             = {rout_casc*1e-6:.3f} Mohm')
print()

print(f'Comparacion:')
print(f'  rout_CE     = {ro1:.0f} ohm  = {ro1*1e-3:.0f} kohm')
print(f'  rout_cascode = {rout_casc:.0f} ohm = {rout_casc*1e-3:.0f} kohm')
print(f'  Mejora = gm*ro = {gm1*ro1:.1f}x')

### Ejercicio 4: Diseno de Buffer CE+CC

**Enunciado:** Se necesita un amplificador con $|A_v| > 10$ y $r_{out} < 500\,\Omega$.
Usar una combinacion CE+CC con $I_C = 500\,\mu A$, $\beta = 200$, $V_A = 10$ V,
$R_S = 100\,\Omega$, $R_L = 2\,k\Omega$. Verificar si cumple las especificaciones.

In [ ]:
# === EJERCICIO 4: Diseno Buffer CE+CC ===
print('=== EJERCICIO 4: Buffer CE+CC ===')
print()

# Datos
IC_4 = 500e-6
beta_4 = 200
VA_4 = 10.0
RS_4 = 100
RL_4 = 2e3

# Parametros
gm_4 = IC_4 / Vt
rpi_4 = beta_4 / gm_4
ro_4 = VA_4 / IC_4

print(f'Parametros: gm={gm_4*1e3:.2f} mA/V, rpi={rpi_4:.0f} ohm, ro={ro_4:.0f} ohm')
print()

# Etapa CE
ro_par_RL_4 = (ro_4 * RL_4) / (ro_4 + RL_4)
Av_ce4 = -(rpi_4 / (rpi_4 + RS_4)) * gm_4 * ro_par_RL_4
rout_ce4 = ro_4

print(f'Etapa CE:')
print(f'  ro||RL = {ro_par_RL_4:.0f} ohm')
print(f'  Av_CE = {Av_ce4:.2f}')
print(f'  rout_CE = {rout_ce4:.0f} ohm')
print()

# Etapa CC (con RS_eff = rout_CE)
RS_eff4 = rout_ce4
Av_cc4 = 1 / (1 + (RS_eff4 + rpi_4) / ((1 + gm_4 * rpi_4) * ro_par_RL_4))
rout_cc4 = (RS_eff4 + rpi_4) / (1 + gm_4 * rpi_4)

print(f'Etapa CC (RS_eff = rout_CE = {RS_eff4:.0f} ohm):')
print(f'  Av_CC = {Av_cc4:.4f}')
print(f'  rout_CC = {rout_cc4:.1f} ohm')
print()

# Resultado final
Av_total = Av_ce4 * Av_cc4
print(f'RESULTADO FINAL:')
print(f'  |Av_total| = {abs(Av_total):.2f}')
print(f'  rout = {rout_cc4:.1f} ohm')
print()

# Verificacion
cumple_av = abs(Av_total) > 10
cumple_rout = rout_cc4 < 500
print(f'Verificacion:')
print(f'  |Av| > 10?      {"SI" if cumple_av else "NO"} (|Av| = {abs(Av_total):.2f})')
print(f'  rout < 500 ohm? {"SI" if cumple_rout else "NO"} (rout = {rout_cc4:.1f} ohm)')

---

## 10. Catalogo de Configuraciones BJT

Referencia rapida de las 6 configuraciones principales.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

configs_cat = [
    {'name': 'CE\n(Emisor Comun)', 'Av': 'Alto (invierte)',
     'rin': '$r_\\pi$', 'rout': '$r_o$', 'color': COLOR_PRINCIPAL,
     'uso': 'Amplificacion'},
    {'name': 'CC\n(Colector Comun)', 'Av': '$\\approx 1$',
     'rin': 'Alto', 'rout': 'Bajo', 'color': COLOR_PUNTO,
     'uso': 'Buffer'},
    {'name': 'CB\n(Base Comun)', 'Av': 'Alto (no invierte)',
     'rin': '$1/g_m$', 'rout': '$g_m r_o^2$', 'color': COLOR_NARANJA,
     'uso': 'Alta frecuencia'},
    {'name': 'Cascode\n(CE + CB)', 'Av': 'Muy alto',
     'rin': '$r_\\pi$', 'rout': '$g_m r_o^2$', 'color': COLOR_ROJO,
     'uso': 'Muy alta ganancia'},
    {'name': 'Buffer\n(CE + CC)', 'Av': 'Alto',
     'rin': '$r_\\pi$', 'rout': 'Muy bajo', 'color': COLOR_MORADO,
     'uso': 'Driving cargas'},
    {'name': 'CE + Carga\nActiva', 'Av': '$-g_m r_o / 2$',
     'rin': '$r_\\pi$', 'rout': '$r_o/2$', 'color': '#8c6d31',
     'uso': 'Alta ganancia IC'},
]

for idx, (ax, cfg) in enumerate(zip(axes.flat, configs_cat)):
    ax.set_xlim(0, 10)
    ax.set_ylim(0, 10)
    ax.axis('off')
    
    # Title box
    ax.add_patch(FancyBboxPatch((0.5, 7), 9, 2.5, boxstyle='round,pad=0.2',
                 facecolor=cfg['color'], edgecolor='white', lw=2, alpha=0.15))
    ax.text(5, 8.5, cfg['name'], ha='center', va='center', fontsize=14,
            fontweight='bold', color=cfg['color'])
    
    # Properties
    props = [
        f"$A_v$: {cfg['Av']}",
        f"$r_{{in}}$: {cfg['rin']}",
        f"$r_{{out}}$: {cfg['rout']}",
        f"Uso: {cfg['uso']}",
    ]
    for i, prop in enumerate(props):
        ax.text(5, 5.5 - i*1.3, prop, ha='center', va='center', fontsize=12,
                color=COLOR_FIJO)
    
    # Border
    ax.add_patch(FancyBboxPatch((0.2, 0.2), 9.6, 9.6, boxstyle='round,pad=0.1',
                 facecolor='none', edgecolor=cfg['color'], lw=2))

plt.suptitle('Catalogo de Configuraciones BJT', fontsize=18, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

---

## 11. Comparacion Final: Amplificadores MOS vs BJT

| Aspecto | MOS | BJT |
|---------|-----|-----|
| $g_m$ para misma $I$ | Menor ($2I_D/V_{EFF}$) | **Mayor** ($I_C/V_T$) |
| Impedancia entrada | $\infty$ (ventaja) | $r_\pi$ finita |
| CS / CE $A_v$ | $-g_m(r_o \| R_L)$ | $-\frac{r_\pi}{r_\pi+R_S} g_m(r_o \| R_L)$ |
| CD / CC $A_v$ | $\approx 1$ | $\approx 1$ (algo menor por $r_\pi$) |
| Cascode $r_{out}$ | $(g_m r_o) r_o$ | $(g_m r_o) r_o$ |
| Consumo de gate/base | Cero (MOS) | $I_B = I_C/\beta$ (BJT) |

> **Conclusion:** El BJT ofrece mayor $g_m$ (y por tanto mayor ganancia) para
> la misma corriente de polarizacion, pero el MOS tiene impedancia de entrada
> infinita. La eleccion depende de la aplicacion.

---

## 12. Resumen de Formulas Clave

### Parametros del BJT
$$g_m = \frac{I_C}{V_T}, \quad r_\pi = \frac{\beta}{g_m}, \quad r_o = \frac{V_A}{I_C}$$

### Emisor Comun (CE)
$$\boxed{A_v = -\frac{r_\pi}{r_\pi + R_S} \cdot g_m \cdot (r_o \| R_L)}$$
$$r_{in} = r_\pi, \quad r_{out} = r_o$$

### Colector Comun (CC)
$$\boxed{A_v \approx \frac{1}{1 + \frac{R_S + r_\pi}{(1 + g_m r_\pi)(r_o \| R_L)}}}$$
$$r_{in} = r_\pi + (1 + g_m r_\pi)(r_o \| R_L), \quad r_{out} = \frac{R_S + r_\pi}{1 + g_m r_\pi}$$

### Cascode (CE + CB)
$$\boxed{r_{out}^{\text{cascode}} = g_m \cdot r_o \cdot r_o}$$

### Carga Activa
$$\boxed{A_v^{\text{activa}} \approx -\frac{g_m \cdot r_o}{2}}$$